# Joint dynamical posteriors: resonance angles and NV2016 variables
Provided by **Trifon Trifonov with the aid of CODEX; use with caution**.

This notebook integrates posterior draws and keeps the **same rows, weights and
stability classifications** for the original parameters, θ summaries and new
NV2016 variables. It exports a combined GUI-readable `.ses`, plots joint
posteriors, and displays the **Ψ cos ψ versus δ** structure diagram.

The defaults are a three-draw, ten-year **software demonstration**, not a research
stability analysis. The input session is preserved. See `README.md` and
[the NV2016 definitions and accuracy limits](../../docs/NV2016_RESONANCE.md).


In [ ]:
from pathlib import Path
import sys
import dill
import numpy as np

# Prefer this checkout when available; a standalone notebook uses pip-installed code.
START_DIRECTORY = Path.cwd().resolve()
ROOT = next((p for p in [START_DIRECTORY, *START_DIRECTORY.parents]
             if (p / 'exostriker/lib/RV_mod/dynamical_posterior.py').is_file()), None)
if ROOT is not None:
    sys.path.insert(0, str(ROOT))
import exostriker
from exostriker.lib.RV_mod.dynamical_posterior import (
    IntegrationOptions, integrate_posterior, export_posterior,
    save_integrated_session, dynamical_columns, diagnostic_label,
    map_nv2016_posterior)
from exostriker.lib.RV_mod.resonance import export_resonance
print('Using Exo-Striker:', exostriker.__file__)

# Replace with your sampled session; absolute paths also work with a pip installation.
SESSION = (ROOT / 'examples/stability/HD128311.ses' if ROOT is not None
           else START_DIRECTORY / 'HD128311.ses')
with SESSION.open('rb') as stream:
    fit = dill.load(stream)
print('Enabled planets (GUI slot, period in days):',
      [(i + 1, fit.P[i]) for i in range(len(fit.use_planet)) if fit.use_planet[i]])


## Select the resonant pair and integration settings

The bundled HD128311 session has **three active planets**; its near-2:1 pair is
**GUI slots 2 and 3**. For TIC279401253, select its session above and set
`PAIR = (1, 2)`. `MMR = (2, 1)` means outer/inner period ratio 2:1.
NV2016 supports first-order resonances k:(k−1).

Increase `COUNT`, duration, and output cadence for research use, and check
convergence. `COUNT = None` integrates every saved posterior row. Set `SOURCE`
to `"ns"` or `"mcmc"` explicitly if both samplers exist. `WORKERS = 1` is a safe
notebook default; larger values use spawned integration processes.

NV2016 is a **planar, low-eccentricity approximation**. For eccentric giant
planets, its analytic separatrix is approximate. The CSV and diagnostics record
model-applicability notes and invariant drift; N-body stability does not imply
resonant membership. ψ is a combination of both θ angles, not either θ itself.


In [ ]:
PAIR = (2, 3)  # HD128311; use (1, 2) for TIC279401253
MMR = (2, 1)
SOURCE = 'ns'
COUNT = 3
WORKERS = 1
SEED = 42

options = IntegrationOptions(
    years=10, timestep_days=1, output_years=0.1,
    timeout_seconds=60, integrator='mvs',  # or REBOUND: 'whfast', 'ias15', 'mercurius', 'saba', 'trace', 'janus'
    max_fractional_a_change=.1, max_eccentricity=.95,
    pair=PAIR, mmr=MMR, nv2016=True, nv2016_points=256)
# Optional HKL outputs: derive_e=True, derive_omega=True, derive_M0=True.
# HKL integration itself is automatic regardless of these output switches.


In [ ]:
result = integrate_posterior(
    fit, options, source=SOURCE, count=COUNT, seed=SEED, workers=WORKERS,
    progress=lambda done, total, record: print(f'{done}/{total}: {record["status"]}'))
print('Accepted posterior weight:', result['weights'][result['stable']].sum())
notes = sorted({note for record in result['records']
                for note in record.get('resonance_nv2016', {}).get('warnings', [])})
for note in notes:
    print('NV2016 model note:', note)


## Export both θ and NV2016 posteriors

Choose a **new** output directory; existing directories are never overwritten.
`posterior.csv/.npz`, `stable_posterior.csv` and the integrated `.ses` include
both θ means/amplitudes and the new NV2016 columns, with aligned posterior
weights and stability labels. `resonance.csv/.npz` additionally include mapped
coordinates and histories; `resonance_structure.pdf/.png` show epoch coordinates.
Initial values, time averages, and amplitudes are separate quantities.


In [ ]:
OUTPUT = START_DIRECTORY / (SESSION.stem + '_dynamical_posterior_nv2016')
export_posterior(result, OUTPUT)
INTEGRATED_SESSION = save_integrated_session(
    fit, result, OUTPUT / (SESSION.stem + '_integrated.ses'))
print('Combined GUI-readable session:', INTEGRATED_SESSION)
print((OUTPUT / 'summary.csv').read_text())


In [ ]:
from IPython.display import Image, display
structure_image = OUTPUT / 'resonance_structure.png'
if structure_image.exists():
    display(Image(filename=str(structure_image)))
else:
    print('No valid NV2016 mappings; inspect resonance.csv for the reasons.')


## Joint θ–NV2016 corner plot

Combine the **original parameter samples and diagnostic columns** first.
`result['samples']` alone contains only the original fitted parameters.
The default joint selection is θ₀ and θ₁ means/amplitudes, δ₀, Ψ₀, ψ₀ and
(Ψ cos ψ)₀. Edit `JOINT_KEYS` below to choose another combination; add original
parameter labels via `EXTRA_ORIGINAL_LABELS` if desired.

θ subscripts in exports are zero-based: θ₀ uses the inner planet's periapse,
θ₁ the outer planet's. With 2:1 these are λ_inner−2λ_outer+ϖ_inner/outer.
Angular means and initial ψ are placed on one continuous circular branch for
plotting; angular amplitudes and the saved posterior values are unchanged.

The gray distribution uses all finite draws, weighted by their posterior
probabilities. The blue overlay uses the stable subset with the same columns
and corresponding weights. At least 100 effective draws are required for each
population; the three-draw demonstration deliberately skips the corner plot.
LaTeX rendering is disabled, and the bundled corner library works with pip installs.


In [ ]:
diagnostic_labels, diagnostics = dynamical_columns(result)
combined_samples = np.column_stack((result['samples'], diagnostics))
combined_labels = list(result['labels']) + list(diagnostic_labels)
pair_prefix = f'pl{PAIR[0]}_pl{PAIR[1]}'
mmr_prefix = pair_prefix + f'_MMR{MMR[0]}_{MMR[1]}'
nv_prefix = pair_prefix + '_NV2016'
JOINT_KEYS = [
    mmr_prefix + '_theta0_mean', mmr_prefix + '_theta0_semiamp',
    mmr_prefix + '_theta1_mean', mmr_prefix + '_theta1_semiamp',
    nv_prefix + '_delta_initial', nv_prefix + '_Psi_initial',
    nv_prefix + '_psi_initial_deg', nv_prefix + '_Psi_cos_psi_initial',
]
EXTRA_ORIGINAL_LABELS = []  # Exact labels from result['labels'], e.g. selected P or e.
joint_labels = [diagnostic_label(key) for key in JOINT_KEYS] + EXTRA_ORIGINAL_LABELS
missing = [label for label in joint_labels if label not in combined_labels]
if missing:
    raise ValueError(f'Missing joint diagnostics: {missing}. Check pair/MMR, NV2016 errors and integration results.')
columns = [combined_labels.index(label) for label in joint_labels]
joint_samples = combined_samples[:, columns].copy()

# Weighted circular branch for θ means and ψ; keep amplitudes and stored data intact.
for col, key in enumerate(JOINT_KEYS):
    if ('_theta' in key and key.endswith('_mean')) or key.endswith('_psi_initial_deg'):
        valid = np.isfinite(joint_samples[:, col]) & (result['weights'] > 0)
        if valid.any():
            z = np.sum(result['weights'][valid] * np.exp(1j * np.radians(joint_samples[valid, col])))
            center = np.degrees(np.angle(z))
            joint_samples[valid, col] = center + (joint_samples[valid, col] - center + 180) % 360 - 180
print('Joint posterior columns:')
for label in joint_labels:
    print(' ', label)


In [ ]:
import matplotlib.pyplot as plt
import corner_ES as corner
plt.rcParams['text.usetex'] = False
MIN_EFFECTIVE_DRAWS = 100
finite = np.isfinite(joint_samples).all(axis=1) & (result['weights'] > 0)

def effective_draws(mask):
    weights = result['weights'][mask]
    return weights.sum() ** 2 / np.sum(weights ** 2) if len(weights) else 0.

if effective_draws(finite) >= MIN_EFFECTIVE_DRAWS:
    plot_range = []
    for col in range(joint_samples.shape[1]):
        lo, hi = np.min(joint_samples[finite, col]), np.max(joint_samples[finite, col])
        padding = max((hi - lo) * 0.02, max(abs(lo), abs(hi), 1.) * 1.e-8)
        plot_range.append((lo - padding, hi + padding))
    fig = corner.corner(
        joint_samples[finite], weights=result['weights'][finite],
        labels=joint_labels, range=plot_range, color='gray',
        plot_datapoints=False, show_titles=True)
    stable = finite & result['stable']
    if effective_draws(stable) >= MIN_EFFECTIVE_DRAWS:
        corner.corner(
            joint_samples[stable], weights=result['weights'][stable],
            labels=joint_labels, range=plot_range, fig=fig,
            color='tab:blue', plot_datapoints=False)
    else:
        print('Stable overlay skipped: fewer than 100 effective stable draws.')
    fig.savefig(OUTPUT / 'joint_theta_nv2016_corner.png', dpi=160, bbox_inches='tight')
    fig.savefig(OUTPUT / 'joint_theta_nv2016_corner.pdf', bbox_inches='tight')
    plt.show()
else:
    print('Increase COUNT for a joint corner plot; effective finite draws:', effective_draws(finite))


## Optional: structure diagram from a saved session, without reintegration

This cell is disabled by default. Set `DO_EPOCH_ONLY = True` and point
`EPOCH_SESSION` at an existing integrated session to map its preserved posterior
rows, weights and stability statuses. An original sampled session also works,
but has no integration-based stability classification. This path exports epoch
coordinates only; use the main integration workflow above for joint θ evolution
summaries and NV2016 time histories.


In [ ]:
DO_EPOCH_ONLY = False
if DO_EPOCH_ONLY:
    EPOCH_SESSION = INTEGRATED_SESSION  # Or Path('/path/to/existing_integrated.ses').
    with EPOCH_SESSION.open('rb') as stream:
        epoch_fit = dill.load(stream)
    epoch_result = map_nv2016_posterior(epoch_fit, options, source=SOURCE, seed=SEED)
    EPOCH_OUTPUT = START_DIRECTORY / (EPOCH_SESSION.stem + '_nv2016_epoch')
    EPOCH_OUTPUT.mkdir(parents=True, exist_ok=False)
    export_resonance(epoch_result, EPOCH_OUTPUT)
    print('Epoch-only output:', EPOCH_OUTPUT)
